# 01 - Inspección inicial

**Proyecto Integrador — Minería de Datos 1**

**Objetivo del proyecto:** Analizar cómo se relaciona el tiempo mensual de visualización con el tipo de plan de suscripción y el país de los usuarios, para identificar patrones de consumo en la plataforma de streaming.

**Propósito de esta etapa:** relevar la estructura general del dataset, sus tipos de datos, valores faltantes, duplicados y primeras observaciones. **No se toman decisiones de limpieza acá** — solo se reúne evidencia que va a justificar las decisiones de la etapa siguiente (`02_calidad_y_limpieza.ipynb`).

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_json('../data/raw/streaming_users_dirty.json')
df.shape

(8160, 8)

## 1. Estructura general

In [2]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head(10)

Filas: 8160
Columnas: 8


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1
5,10005,20,Básico,670.2,Uruguay,Drama,2020-07-03,2
6,10006,37,Básico,346.6,Perú,Thriller,2019-07-26,1
7,10007,31,Estándar,974.6,Chile,Acción,2019-02-24,1
8,10008,36,Premium,1432.2,Colombia,Romance,2025-08-03,2
9,10009,37,Estándar,1375.4,Argentina,Thriller,2024-02-12,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 756.7 KB


**Observación:** las columnas tienen los tipos que esperaríamos a primera vista (`int64`, `float64`, `object`), pero eso no garantiza que los *valores* dentro de cada columna sean consistentes. Lo vamos a chequear columna por columna en la sección 3.

## 2. Valores faltantes y duplicados

In [4]:
nulos = df.isna().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
pd.DataFrame({'nulos': nulos, 'pct_nulos': pct_nulos}).sort_values('nulos', ascending=False)

,nulos,pct_nulos
last_login_date,320,3.92
favorite_genre,240,2.94
monthly_watch_time_mins,193,2.37
user_id,0,0.00
subscription_plan,0,0.00
age,0,0.00
country,0,0.00
customer_support_tickets,0,0.00


In [5]:
dup_user_id = df['user_id'].duplicated().sum()
dup_filas_completas = df.duplicated().sum()
print(f"user_id duplicados: {dup_user_id}")
print(f"Filas 100% duplicadas: {dup_filas_completas}")

user_id duplicados: 160
Filas 100% duplicadas: 126


**Observación:**
- `favorite_genre` tiene nulos (~3%).
- `last_login_date` tiene una proporción más alta de nulos (~4%), y además vamos a ver en la sección 3 que tiene valores que *parecen* fechas pero no lo son (`"not_available"`, `"0000-00-00"`).
- Hay `user_id` repetidos y filas completamente duplicadas. Esto sugiere que en algún punto del proceso de generación/carga de datos se insertaron registros repetidos — es evidencia a tratar en la etapa de limpieza, no algo que se resuelve acá.

## 3. Inspección columna por columna

In [6]:
for col in df.columns:
    print(f"--- {col} ({df[col].dtype}) ---")
    print(f"  Valores únicos: {df[col].nunique(dropna=True)}")
    if df[col].dtype == 'object':
        print(f"  Ejemplos: {df[col].dropna().unique()[:8]}")
    print()

--- user_id (int64) ---
  Valores únicos: 8000

--- age (int64) ---
  Valores únicos: 69

--- subscription_plan (str) ---
  Valores únicos: 15

--- monthly_watch_time_mins (float64) ---
  Valores únicos: 5788

--- country (str) ---
  Valores únicos: 26

--- favorite_genre (str) ---
  Valores únicos: 28

--- last_login_date (str) ---
  Valores únicos: 3062

--- customer_support_tickets (int64) ---
  Valores únicos: 9



### 3.1 Variables categóricas: `subscription_plan`, `country`, `favorite_genre`

In [7]:
df['subscription_plan'].value_counts(dropna=False)

subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64

In [8]:
df['country'].value_counts(dropna=False)

country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
MEX             16
Chile           16
argentina       16
PER             16
chile           15
Mexico          15
Peru            15
BRA             15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64

In [9]:
df['favorite_genre'].value_counts(dropna=False)

favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Thriller       1090
Romance        1090
Acción         1082
Crime          1067
NaN             240
Action           22
COMEDIA          19
Crimen           18
CRIME            17
Romance          17
DRAMA            16
Documentary      16
Comedia          15
DOC              15
ACCIÓN           14
THRILLER         14
ROMANCE          14
comedy           12
Thriller         12
romance          12
drama            10
documental       10
accion            8
Drama             7
thriler           6
crime             5
Name: count, dtype: int64

**Observación clave:** las tres variables categóricas tienen *muchos más valores únicos de los que deberían existir conceptualmente*. Por ejemplo, `subscription_plan` debería tener 3 categorías (Básico / Estándar / Premium) pero aparecen 15 variantes distintas por diferencias de mayúsculas/minúsculas, espacios en blanco, tildes, abreviaturas (`Std`) y anglicismos (`Basic`, `Premiun`). Lo mismo pasa con `country` (nombres en inglés, códigos ISO3 como `COL`/`ARG`, variaciones de casing) y con `favorite_genre`. Esto es evidencia directa de que se necesita una etapa de normalización de texto antes de poder agrupar o graficar por estas variables — que es exactamente lo que pide nuestro objetivo (agrupar por plan y por país).

### 3.2 Variables numéricas: `age`, `monthly_watch_time_mins`, `customer_support_tickets`

In [10]:
df[['age', 'monthly_watch_time_mins', 'customer_support_tickets']].describe()

,age,monthly_watch_time_mins,customer_support_tickets
count,8160.000000,7967.000000,8160.000000
mean,34.096814,1107.346153,1.800980
std,14.511304,5310.442604,11.334969
min,-5.000000,-120.000000,-1.000000
25%,25.000000,489.200000,0.000000
50%,33.000000,757.400000,1.000000
75%,42.000000,1045.700000,1.000000
max,150.000000,99999.000000,150.000000


In [11]:
print("Edades <=0 o >100:", ((df['age'] <= 0) | (df['age'] > 100)).sum())
print(df.loc[(df['age'] <= 0) | (df['age'] > 100), 'age'].value_counts().sort_index())

Edades <=0 o >100: 98
age
-5      21
 0      24
 130    34
 150    19
Name: count, dtype: int64


In [12]:
print("monthly_watch_time_mins == NaN:", df['monthly_watch_time_mins'].isna().sum())
print("monthly_watch_time_mins < 0:", (df['monthly_watch_time_mins'] < 0).sum())
print("monthly_watch_time_mins == 99999.0 (posible valor centinela):", (df['monthly_watch_time_mins'] == 99999.0).sum())
df['monthly_watch_time_mins'].describe()

monthly_watch_time_mins == NaN: 193
monthly_watch_time_mins < 0: 49
monthly_watch_time_mins == 99999.0 (posible valor centinela): 20


count     7967.000000
mean      1107.346153
std       5310.442604
min       -120.000000
25%        489.200000
50%        757.400000
75%       1045.700000
max      99999.000000
Name: monthly_watch_time_mins, dtype: float64

In [13]:
print("customer_support_tickets < 0:", (df['customer_support_tickets'] < 0).sum())
print("customer_support_tickets > 50 (sospechoso):", (df['customer_support_tickets'] > 50).sum())
df['customer_support_tickets'].value_counts().sort_index()

customer_support_tickets < 0: 29
customer_support_tickets > 50 (sospechoso): 67


customer_support_tickets
-1        29
 0      3628
 1      2885
 2      1179
 3       285
 4        73
 5        14
 99       35
 150      32
Name: count, dtype: int64

**Observación:**
- `age` tiene valores imposibles: negativos (-5), cero, y valores como 130 o 150 que no son edades humanas plausibles.
- `monthly_watch_time_mins` (la variable central de nuestro objetivo) tiene valores `NaN`, valores negativos (imposibles para un tiempo de visualización) y un grupo de exactamente **20 registros en 99999.0**, un valor muy por encima del resto de la distribución y repetido de forma idéntica — patrón típico de un *valor centinela* usado para marcar datos faltantes o error, no una medición real.
- `customer_support_tickets` tiene valores negativos (-1) y un grupo de valores en 150, muy alejados del resto — mismo patrón de posible centinela.

### 3.3 `last_login_date`

In [14]:
df['last_login_date'].value_counts(dropna=False).head(15)

last_login_date
NaN              320
2026-15-03        20
0000-00-00        17
2029-01-01        15
not_available     14
31-02-2022        13
2019-01-17        10
2018-08-18         9
2025-09-29         9
2024-08-29         9
2023-02-02         8
2020-10-13         8
2025-05-05         8
2023-02-06         8
2018-11-20         8
Name: count, dtype: int64

In [15]:
import re
patrones = {'YYYY-MM-DD': r'^\d{4}-\d{2}-\d{2}$',
            'MM-DD-YYYY': r'^\d{2}-\d{2}-\d{4}$',
            'YYYY/MM/DD': r'^\d{4}/\d{2}/\d{2}$'}

def clasificar(v):
    if pd.isna(v):
        return 'nulo'
    for nombre, pat in patrones.items():
        if re.match(pat, str(v)):
            return nombre
    return 'otro/inválido'

df['last_login_date'].apply(clasificar).value_counts()

last_login_date
YYYY-MM-DD       7428
nulo              320
MM-DD-YYYY        265
YYYY/MM/DD        133
otro/inválido      14
Name: count, dtype: int64

**Observación:** `last_login_date` mezcla al menos 3 formatos de fecha distintos, valores nulos, y strings que no representan fechas (`"not_available"`, `"0000-00-00"`). Hay que unificar el formato antes de poder usar esta columna (si el grupo decide usarla — no es una variable central para nuestro objetivo, que se enfoca en tiempo de visualización, plan y país).

## 4. Resumen de hallazgos para la etapa de limpieza

| Campo | Hallazgo | A resolver en 02_calidad_y_limpieza |
|---|---|---|
| `user_id` | 160 duplicados, 126 filas 100% duplicadas | Eliminar duplicados |
| `subscription_plan` | 15 variantes de texto para 3 categorías reales | Normalizar texto (trim, minúsculas, mapeo de abreviaturas) |
| `country` | 26 variantes de texto para 7 países reales | Normalizar texto y códigos ISO3 |
| `favorite_genre` | 29 variantes + ~3% nulos | Normalizar texto; decidir tratamiento de nulos (no es variable central del objetivo) |
| `age` | Rango -5 a 150 | Definir rango plausible y tratar valores fuera de rango |
| `monthly_watch_time_mins` | NaN, negativos, centinela en 99999.0 | Tratar NaN y valores imposibles (variable central del objetivo) |
| `customer_support_tickets` | Negativos, posible centinela en 150 | Definir rango plausible |
| `last_login_date` | 3+ formatos, nulos, strings inválidos | Unificar formato o descartar si no es central |

Estos hallazgos son la base de las decisiones que se van a justificar, aplicar y registrar (en `logs/pipeline_log.csv`) en la siguiente notebook.